In [ ]:
#Deployment

role = "your-role"
region = "us-east-1"
BUCKET = "your-bucket-name"

In [ ]:
===========Load the Model===================

In [ ]:
import sagemaker
from sagemaker.model import Model
BUCKET = "your-bucket-name"
sess = sagemaker.Session()
role = role
model_artifact ="your-job-path-name"
model = Model(
    model_data=model_artifact,
    image_uri=sagemaker.image_uris.retrieve(
        framework="xgboost",
        region=sess.boto_region_name,
        version="1.7-1"
    ),
    role=role,
    sagemaker_session=sess
)

In [ ]:
=============Enable the Data Capture==================

In [ ]:
from sagemaker.model_monitor import DataCaptureConfig
import sagemaker
from sagemaker.model_monitor import ModelMonitor

BUCKET = "your-bucket-name"

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f"s3://{BUCKET}/data-capture"
)

In [ ]:
=============Deploy thr Model to endpoint==================

In [ ]:

endpoint_name = "your-endpoint-name"

# ----------------------------------------
# 3️⃣ Deploy the model
# ----------------------------------------
# Assume 'model' object already exists from training
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config  # optional
)

In [ ]:
=================Test the Endpoint if it is working or not===============

In [ ]:
#Test the Endpoint

import pandas as pd

test_s3_path = "test/test_predict.csv"

test_df = pd.read_csv(test_s3_path)

# Take only one row
sample = test_df.iloc[1]

In [ ]:
from sagemaker.predictor import Predictor

# Use your endpoint name
endpoint_name = "your-endpoint-name"

# Create Predictor and set CSV input/output
predictor = Predictor(endpoint_name=endpoint_name)
predictor.content_type = "text/csv"  # tells the predictor to send CSV
predictor.accept = "text/csv"        # expects CSV response

# Test the predictor object
print(predictor)

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

result = predictor.predict(sample)
print("Prediction:", result)

In [ ]:
================Create a Baseline using train data========================

In [ ]:
import boto3
import pandas as pd

BUCKET = "bucket-name"
KEY = "batch_input/train/train_sm.csv"

s3 = boto3.client("s3")

# Download file locally
s3.download_file(BUCKET, KEY, "train_sm.csv")

print("Downloaded successfully!")

In [ ]:
import pandas as pd

df = pd.read_csv("train_sm.csv", header=None)

# Drop first column (target: sales)
df = df.iloc[:, 1:]

# Create proper feature names
df.columns = [f"feature_{i}" for i in range(df.shape[1])]

df.to_csv("train_fixed.csv", index=False)

In [ ]:
print(df.columns)

In [ ]:
df.shape

In [ ]:
import boto3

# Create S3 client
s3 = boto3.client("s3")

# Upload file
s3.upload_file(
    Filename="train_fixed.csv",  # local file name
    Bucket="your-bucket-name",
    Key="batch_input/train/train_fixed.csv"
)

print("Upload successful ✅")

In [ ]:
from sagemaker.model_monitor import DefaultModelMonitor
BUCKET="your-bucket-name"
monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600
)

baseline_job = monitor.suggest_baseline(
    baseline_dataset="s3://{BUCKET}/batch_input/train/train_fixed.csv",
    dataset_format={"csv": {"header": True}},
    output_s3_uri=f"s3://{BUCKET}/baseline",
    wait=True
)

In [ ]:
baseline_job.describe()

In [ ]:
from sagemaker.model_monitor import BaseliningJob

# Use the latest baselining job from the monitor object
baseline_job = monitor.latest_baselining_job

# Get S3 URIs for statistics and constraints
stats_s3 = baseline_job.baseline_statistics()
constraints_s3 = baseline_job.suggested_constraints()

print("Statistics JSON:", stats_s3)
print("Constraints JSON:", constraints_s3)

In [ ]:
=================Data Monitor============================

In [ ]:
from sagemaker.model_monitor import EndpointInput

endpoint_input = EndpointInput(
    endpoint_name="your-edpoint-name",
    destination="",
    #content_type="text/csv",  # tells monitor the endpoint output is CSV
    #input_mode="File"          # ensures the data is passed as a file
)

monitor.create_monitoring_schedule(
    monitor_schedule_name="your-monitor-name",
    endpoint_input=endpoint_input,
    output_s3_uri=f"s3://{BUCKET}/monitoring-output",
    statistics=stats_s3,
    constraints=constraints_s3,
    schedule_cron_expression="cron(0 * ? * * *)"  # runs every hour
)

In [ ]:
import pandas as pd
import json

train_monitor = pd.read_csv("train_monitor.csv")
prod_monitor = pd.read_csv("prod_monitor.csv")

with open("feature_columns.json") as f:
    feature_cols = json.load(f)

print("Features:", feature_cols)

In [ ]:
================Send production Data to the Endoint============================

In [ ]:
import boto3

runtime = boto3.client("sagemaker-runtime")
endpoint_name = "your-endpoint-name"

batch_size = 200
all_preds = []

for i in range(0, len(prod_monitor), batch_size):
    batch = prod_monitor.iloc[i:i+batch_size][feature_cols]

    payload = batch.to_csv(index=False, header=False)

    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="text/csv",
        Body=payload
    )

    result = response["Body"].read().decode("utf-8").strip().split("\n")
    all_preds.extend([float(x) for x in result])

print("Total predictions:", len(all_preds))

In [ ]:
prod_monitor["prediction"] = all_preds

print(prod_monitor.head())
print(prod_monitor.shape)

In [ ]:
prod_monitor.to_csv("prod_monitor_with_predictions.csv", index=False)
print("Saved prod_monitor_with_predictions.csv")

In [ ]:
===========Prepare reference and current data for drift checking==================

In [ ]:
reference_df = train_monitor[feature_cols].copy()
current_df = prod_monitor[feature_cols].copy()

print("Reference shape:", reference_df.shape)
print("Current shape:", current_df.shape)

In [ ]:
reference_df.columns

In [ ]:
==========Data drift(feature drift)==============

In [ ]:
import numpy as np
import pandas as pd

candidate_features_mon = [
    "year","month", "day", "is_weekend",
    "onpromotion", "dcoilwtico",
    "transactions", "is_holiday","is_earthquake_period"
]

FEATURES_MON = [c for c in candidate_features_mon if c in reference_df.columns]
print("Monitoring features:", FEATURES_MON)

In [ ]:
def psi(expected, actual, bins=10):
    expected = np.asarray(expected)
    actual = np.asarray(actual)

    if np.nanstd(expected) == 0 and np.nanstd(actual) == 0:
        return 0.0

    expected_clean = expected[~np.isnan(expected)]
    actual_clean = actual[~np.isnan(actual)]

    if len(expected_clean) == 0 or len(actual_clean) == 0:
        return np.nan

    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(expected_clean, quantiles))

    if len(edges) < 3:
        return 0.0

    exp_counts, _ = np.histogram(expected_clean, bins=edges)
    act_counts, _ = np.histogram(actual_clean, bins=edges)

    exp_perc = exp_counts / max(exp_counts.sum(), 1)
    act_perc = act_counts / max(act_counts.sum(), 1)

    exp_perc = np.where(exp_perc == 0, 1e-6, exp_perc)
    act_perc = np.where(act_perc == 0, 1e-6, act_perc)

    return np.sum((act_perc - exp_perc) * np.log(act_perc / exp_perc))

In [ ]:
drift_report = []

for f in FEATURES_MON:
    psi_val = psi(
        reference_df[f].fillna(0).values,
        current_df[f].fillna(0).values,
        bins=10
    )
    drift_report.append((f, psi_val))

drift_df = pd.DataFrame(drift_report, columns=["feature", "psi"])
drift_df = drift_df.sort_values("psi", ascending=False)

print(drift_df)

In [ ]:
def psi_status(x):
    if pd.isna(x):
        return "unknown"
    elif x < 0.10:
        return "stable"
    elif x < 0.20:
        return "moderate drift"
    else:
        return "significant drift"

drift_df["status"] = drift_df["psi"].apply(psi_status)
print(drift_df)

In [ ]:
drift_df.to_csv("feature_drift_report.csv", index=False)
print("Saved feature_drift_report.csv")

In [ ]:
==============Prediction Drift==========================
#Sending training data to endpoint to store predictions as baseline predictions and then compare them with  production predictions

In [ ]:
baseline_preds = []

for i in range(0, len(reference_df), batch_size):
    batch = reference_df.iloc[i:i+batch_size][feature_cols]

    payload = batch.to_csv(index=False, header=False)

    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="text/csv",
        Body=payload
    )

    result = response["Body"].read().decode("utf-8").strip().split("\n")
    baseline_preds.extend([float(x) for x in result])

print("Baseline predictions:", len(baseline_preds))

In [ ]:
=========Calculate prediction drift with train predictions(baseline) and production data predictions================

In [ ]:
pred_psi = psi(np.array(baseline_preds), np.array(all_preds), bins=10)

print("Prediction drift PSI:", pred_psi)
print("Prediction drift status:", psi_status(pred_psi))

In [ ]:
#. Save prediction drift report
prediction_drift_df = pd.DataFrame({
    "metric": ["prediction_psi"],
    "value": [pred_psi],
    "status": [psi_status(pred_psi)]
})

prediction_drift_df.to_csv("prediction_drift_report.csv", index=False)
print(prediction_drift_df)

In [ ]:
import boto3

sm = boto3.client("sagemaker")

endpoint_name = "your-endpoint-name"

sm.delete_endpoint(EndpointName=endpoint_name)

print("Endpoint deletion started")

In [ ]:
sm.delete_monitoring_schedule(
    MonitoringScheduleName="your-monitor-name"
)

print("Monitoring schedule deletion started")